In [2]:
# Humor Detection using Ollama Models

## Imports

# ```python
import pandas as pd
import ollama

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

tqdm.pandas()
# ```

## Configuration

# ```python
DATA_PATH = "Data/testing/joke_categorization(testing).csv"

LABELS = [
    "clean",
    "dark",
    "dirty"
]

MODELS = [
    # "qwen2.5",
    # "llama3.1",
    # "mistral",
    # "gemma3",
    # "command-r7b-arabic",
    "deepseek-r1"
]
# ```

## Prompt

# ```python
def build_prompt(text):

    return f"""
You are an expert in humor analysis.

Task:
Classify the joke into one of the following humor categories.

Labels:
- clean: Harmless humor suitable for all audiences. No offensive, sexual, or disturbing content.
- dark: Humor involving sensitive topics such as death, illness, tragedy, violence, or other disturbing themes.
- dirty: Humor involving sexual content, explicit language, vulgarity, or obscene topics.

Rules:
- Choose exactly one label.
- Return ONLY the label.
- Do not provide explanations.

Joke:
{text}
"""
# ```

## Prediction Function

# ```python
def predict_label(text, model_name):

    try:

        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text)
                }
            ]
        )

        prediction = response["message"]["content"].strip()

        if prediction in LABELS:
            return prediction

        prediction = prediction.lower()

        for label in LABELS:
            if prediction == label.lower():
                return label

        return "INVALID"

    except Exception:
        return "INVALID"
# ```

## Evaluation

# ```python
df = pd.read_csv(DATA_PATH)

results = []

for model_name in MODELS:

    preds = []

    for text in tqdm(
        df["text"],
        total=len(df),
        desc=model_name
    ):
        preds.append(
            predict_label(text, model_name)
        )

    acc = accuracy_score(df["label"], preds)

    f1 = f1_score(
        df["label"],
        preds,
        average="macro"
    )

    print("\n", "="*60)
    print(model_name)
    print("Accuracy:", round(acc,4))
    print("Macro F1:", round(f1,4))

    print(
        classification_report(
            df["label"],
            preds,
            digits=4
        )
    )

    tmp = df.copy()
    tmp["prediction"] = preds

    tmp.to_csv(
        f"predictions_{model_name}_categorization.csv",
        index=False
    )

    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "macro_f1",
    ascending=False
)
# ```


deepseek-r1:   0%|          | 0/302 [00:00<?, ?it/s]


deepseek-r1
Accuracy: 0.7185
Macro F1: 0.306
              precision    recall  f1-score   support

     INVALID     0.0000    0.0000    0.0000         0
       clean     0.9279    0.7630    0.8374       270
        dark     0.2500    0.5263    0.3390        19
       dirty     0.0345    0.0769    0.0476        13

    accuracy                         0.7185       302
   macro avg     0.3031    0.3416    0.3060       302
weighted avg     0.8468    0.7185    0.7720       302



/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,model,accuracy,macro_f1
0,deepseek-r1,0.718543,0.306


In [3]:
import pandas as pd
import ollama

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

tqdm.pandas()


DATA_PATH = "Data/testing/joke_categorization(testing).csv"
TRAIN_PATH = "Data/training/joke_categorization(training).csv"

LABELS = [
    "clean",
    "dark",
    "dirty"
]

MODELS = [
    # "qwen2.5",
    # "llama3.1",
    # "mistral",
    # "gemma3",
    # "command-r7b-arabic",
    "deepseek-r1"
]

N_SHOTS = 4

df_test = pd.read_csv(DATA_PATH)
df_train = pd.read_csv(TRAIN_PATH)

def build_few_shot_examples(train_df, label, n_shots=2):

    subset = train_df[train_df["label"] == label]

    if len(subset) == 0:
        raise ValueError(f"No training samples found for label: {label}")

    sampled = subset.sample(
        min(n_shots, len(subset)),
        random_state=42
    )

    return [
        (row["text"], row["label"])
        for _, row in sampled.iterrows()
    ]

def build_prompt(text, few_shot_examples):

    example_text = "\n\n".join([
        f"""Text:
{ex_text}

Label:
{ex_label}"""
        for ex_text, ex_label in few_shot_examples
    ])

    return f"""
You are an expert in humor analysis.

Task:
Classify the joke into one of the following humor categories.

Labels:
- clean: Harmless humor suitable for all audiences. No offensive, sexual, or disturbing content.
- dark: Humor involving sensitive topics such as death, illness, tragedy, violence, or other disturbing themes.
- dirty: Humor involving sexual content, explicit language, vulgarity, or obscene topics.

Rules:
- Choose exactly one label.
- Return ONLY the label.
- Do not provide explanations.

Examples:
{example_text}

Now classify this:

Text:
{text}

Label:
"""


def predict_label(text, model_name, few_shot_examples):

    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text, few_shot_examples)
                }
            ]
        )

        prediction = response["message"]["content"].strip()

        if prediction in LABELS:
            return prediction

        prediction = prediction.lower()

        for label in LABELS:
            if prediction == label.lower():
                return label

        return "INVALID"

    except Exception:
        return "INVALID"

results = []

for model_name in MODELS:

    preds = []

    for _, row in tqdm(
        df_test.iterrows(),
        total=len(df_test),
        desc=model_name
    ):

        text = row["text"]
        true_label = row["label"]

        # 🔥 CLASS-CONDITIONED FEW-SHOT (oracle)
        few_shot_examples = build_few_shot_examples(
            df_train,
            label=true_label,
            n_shots=N_SHOTS
        )

        preds.append(
            predict_label(text, model_name, few_shot_examples)
        )

    acc = accuracy_score(df_test["label"], preds)

    f1 = f1_score(
        df_test["label"],
        preds,
        average="macro"
    )

    print("\n", "="*60)
    print(model_name)
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(f1, 4))

    print(
        classification_report(
            df_test["label"],
            preds,
            digits=4
        )
    )

    tmp = df_test.copy()
    tmp["prediction"] = preds

    tmp.to_csv(
        f"predictions_class_conditioned_fewshot_{model_name}_categorization.csv",
        index=False
    )

    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values("macro_f1", ascending=False)




deepseek-r1:   0%|          | 0/302 [00:00<?, ?it/s]


deepseek-r1
Accuracy: 0.9007
Macro F1: 0.6714
              precision    recall  f1-score   support

       clean     0.9583    0.9370    0.9476       270
        dark     0.5714    0.6316    0.6000        19
       dirty     0.4118    0.5385    0.4667        13

    accuracy                         0.9007       302
   macro avg     0.6472    0.7024    0.6714       302
weighted avg     0.9105    0.9007    0.9050       302



,model,accuracy,macro_f1
0,deepseek-r1,0.900662,0.671411
